**OBSOLETE -- superseded by the 2026-08-25 reorientation plan.**

Exploratory analysis already done; its findings are folded into the
project's memory/README, not something this notebook needs to keep
providing. Kept as a historical record, not maintained going forward.
See README.md for the current plan.

# 01 - EDA DICOM

Objetivo (plan Fase 1): estructura de planos/series por estudio, tamano/espaciado de voxel, orden fisico real de los cortes (NO por nombre de archivo - ver src/data.py::load_dicom_series), y si los tres planos (sagital/coronal/axial) estan siempre presentes.

Referencias: docs.monai.io, pydicom, y los dos notebooks de referencia en data/raw/_reference_kernels/.

In [ ]:
# NOTA: este notebook esta pensado para correr como Kaggle Notebook
# (kernel) contra el dataset montado, no en local (0.5 TB no cabe aqui
# - ver README.md). Sync notebook<->Kaggle es manual por ahora, asi que
# las constantes de src/config.py se repiten aqui a mano; mantenerlas
# sincronizadas si config.py cambia.

import os
import random
from collections import Counter
from pathlib import Path

import numpy as np
import pandas as pd
import pydicom

# Confirmado 2026-08-16: Kaggle monta las competiciones bajo
# /kaggle/input/competitions/<slug>/, no /kaggle/input/<slug>/ a secas.
RAW_DIR = Path("/kaggle/input/competitions/rsna-knee-abnormality-detection")
assert RAW_DIR.exists(), f"Dataset no montado en {RAW_DIR} - anadelo como input del kernel."

# Copiado de src/config.py::OFFICIAL_LABEL_COLUMNS / FINDINGS / MRI_PLANES.
OFFICIAL_LABEL_COLUMNS = {
    "acl_injury": "ACL",
    "mcl_injury": "MCL",
    "medial_meniscus_tear": "Medial Meniscus",
    "lateral_meniscus_tear": "Lateral Meniscus",
    "oa_medial_compartment": "Medial OA",
    "oa_lateral_compartment": "Lateral OA",
    "oa_patellofemoral_compartment": "PF OA",
    "effusion": "Effusion",
    "synovitis": "Synovitis",
    "bakers_cyst": "Baker's",
    "bone_contusion": "Contusion",
    "fracture": "Fracture",
}
FINDINGS = list(OFFICIAL_LABEL_COLUMNS.keys())
LABEL_COLS = list(OFFICIAL_LABEL_COLUMNS.values())
MRI_PLANES = ("sagittal", "coronal", "axial")

random.seed(42)
np.random.seed(42)


## A. `train.csv` — schema y patron de nulos del gold subset

Objetivo: confirmar que las 12 columnas de `OFFICIAL_LABEL_COLUMNS` existen, y que el "gold subset" (~58 estudios) corresponde a filas con las 12 etiquetas pobladas — no a un patron parcial (algunas etiquetas si, otras no).

In [ ]:
train = pd.read_csv(RAW_DIR / "train.csv")
print("train.csv shape:", train.shape)
print("columns:", list(train.columns))

expected_cols = ["StudyInstanceUID", "Report"] + LABEL_COLS
missing_cols = [c for c in expected_cols if c not in train.columns]
print("Missing expected columns:", missing_cols)

# Gold = filas con las 12 etiquetas pobladas (no-nulas).
n_labels_present = train[LABEL_COLS].notna().sum(axis=1)
print("\nDistribucion de cuantas de las 12 etiquetas estan pobladas, por fila:")
print(n_labels_present.value_counts().sort_index())

gold_mask = n_labels_present == len(LABEL_COLS)
print(f"\nEstudios con las 12 etiquetas pobladas (gold): {gold_mask.sum()} / {len(train)}")

# Si el patron NO es all-or-nothing (hay filas con 1-11 etiquetas), esto
# rompe el supuesto "gold = ~58 estudios completos" del README y hay que
# revisarlo antes de escribir src/data.py::load_gold_labels.
partial = n_labels_present[(n_labels_present > 0) & (n_labels_present < len(LABEL_COLS))]
print(f"Estudios con etiquetado PARCIAL (ni 0 ni 12): {len(partial)}")

print("\nReport nulo/vacio en:", train["Report"].isna().sum(), "de", len(train), "filas")


## B. `test.csv` — confirmar ausencia de `Report`

Hecho clave del diseno (README.md): el texto del informe solo existe en train, nunca en test — por eso `src/labelers.py` es solo fuente de etiquetas de entrenamiento, nunca input del modelo en inferencia.

In [ ]:
test = pd.read_csv(RAW_DIR / "test.csv")
print("test.csv shape (recordar: es solo un ejemplo, ~1300 estudios reales en scoring):", test.shape)
print("columns:", list(test.columns))
assert "Report" not in test.columns, "Report aparece en test.csv - contradice la Dataset Description."
print("OK: Report ausente en test.csv")


## C. `train_series.csv` / `test_series.csv` — planos y `Fluid_Sensitive` vs `Fat_Suppression`

Objetivo (plan Fase 1): ¿estan los 3 planos siempre presentes por estudio? Y medir la tasa real de acuerdo entre `Fluid_Sensitive`/`Fat_Suppression` — la Dataset Description oficial dice que NO son siempre equivalentes (contradice al notebook pilkwang, que vio 100% de acuerdo en su muestra).

In [ ]:
train_series = pd.read_csv(RAW_DIR / "train_series.csv")
test_series = pd.read_csv(RAW_DIR / "test_series.csv")
print("train_series.csv shape:", train_series.shape, "| columns:", list(train_series.columns))
print("test_series.csv shape:", test_series.shape)

print("\nAnatomical_Plane value_counts:")
print(train_series["Anatomical_Plane"].value_counts())

# Planos presentes por estudio.
planes_per_study = (
    train_series.groupby("StudyInstanceUID")["Anatomical_Plane"]
    .apply(lambda s: frozenset(p.lower() for p in s))
)
all_three = planes_per_study.apply(lambda s: s == frozenset(MRI_PLANES))
print(f"\nEstudios con los 3 planos presentes: {all_three.sum()} / {len(planes_per_study)}")
print("Patrones de planos mas comunes (frozenset -> n_estudios):")
for pattern, n in Counter(planes_per_study).most_common(10):
    print(f"  {sorted(pattern)}: {n}")

# Series por estudio (media/mediana/max) - relevante para el diseno de "slots" en src/model.py.
series_per_study = train_series.groupby("StudyInstanceUID").size()
print(f"\nSeries por estudio: media={series_per_study.mean():.1f}, mediana={series_per_study.median()}, max={series_per_study.max()}")

# Fluid_Sensitive vs Fat_Suppression: tasa real de acuerdo (NO asumir que son iguales).
agree = train_series["Fluid_Sensitive"] == train_series["Fat_Suppression"]
print(f"\nFluid_Sensitive == Fat_Suppression en {agree.mean():.4%} de las filas ({(~agree).sum()} desacuerdos de {len(agree)})")
if (~agree).any():
    print("Ejemplos de desacuerdo:")
    print(train_series.loc[~agree, ["StudyInstanceUID", "SeriesInstanceUID", "Anatomical_Plane", "Fluid_Sensitive", "Fat_Suppression"]].head(10))


## D. Headers DICOM — allowlist de 86 tags y orden fisico de cortes

Critico para `src/data.py::load_dicom_series`: la Dataset Description dice que cada DICOM fue recortado a un allowlist de 86 tags. Hay que confirmar que `InstanceNumber` y/o `ImagePositionPatient` sobrevivieron ese recorte — si ninguno esta, la estrategia de ordenar por posicion fisica (en vez de por nombre de archivo, rho~0.01 segun los notebooks de referencia) no es viable tal como esta documentada y hay que buscar una alternativa.

In [ ]:
sample_study = train_series["StudyInstanceUID"].iloc[0]
sample_series = train_series.loc[
    train_series["StudyInstanceUID"] == sample_study, "SeriesInstanceUID"
].iloc[0]
series_dir = RAW_DIR / "train_series" / sample_study / sample_series
dcm_files = sorted(series_dir.glob("*.dcm"))  # orden por NOMBRE - solo para comparar mas abajo
print(f"Serie de muestra: {sample_study}/{sample_series} -> {len(dcm_files)} cortes")

ds0 = pydicom.dcmread(dcm_files[0])
present_tags = sorted(e.keyword for e in ds0 if e.keyword)
print(f"\n{len(present_tags)} tags con nombre presentes en este archivo (el allowlist oficial dice 86):")
print(present_tags)

print("\nInstanceNumber presente:", "InstanceNumber" in present_tags)
print("ImagePositionPatient presente:", "ImagePositionPatient" in present_tags)
print("PixelSpacing presente:", "PixelSpacing" in present_tags)
print("TransferSyntaxUID:", getattr(ds0.file_meta, "TransferSyntaxUID", "?"))


In [ ]:
# Replica del check de los notebooks de referencia: orden por nombre de
# archivo vs. InstanceNumber vs. SliceLocation, en varias series (no
# solo una) para que el resultado no dependa de un caso suelto.
#
# OJO: ImagePositionPatient es un punto 3D (x,y,z); el eje que varia
# entre cortes depende de la orientacion del plano (sagital/coronal/
# axial usan ejes distintos), asi que proyectar a pelo sobre un indice
# fijo (p.ej. [2], pensando en "z") esta MAL en planos no-axiales -
# produce una columna constante (visto como ConstantInputWarning mas
# abajo en al menos una serie). SliceLocation es el escalar que DICOM
# ya calcula para esto (proyeccion sobre la normal del plano), asi que
# se usa eso en vez de indexar ImagePositionPatient a mano.
from scipy.stats import spearmanr

sample_series_keys = (
    train_series[["StudyInstanceUID", "SeriesInstanceUID"]]
    .drop_duplicates()
    .sample(n=min(10, len(train_series)), random_state=42)
)

for _, row in sample_series_keys.iterrows():
    d = RAW_DIR / "train_series" / row["StudyInstanceUID"] / row["SeriesInstanceUID"]
    files = sorted(d.glob("*.dcm"))
    if len(files) < 3:
        continue
    inst_nums, slice_locs = [], []
    for f in files:
        ds = pydicom.dcmread(f, stop_before_pixels=True)
        inst_nums.append(int(ds.InstanceNumber) if "InstanceNumber" in ds else np.nan)
        slice_locs.append(float(ds.SliceLocation) if "SliceLocation" in ds else np.nan)
    filename_rank = np.arange(len(files))
    rho_inst = spearmanr(filename_rank, inst_nums).correlation if not np.isnan(inst_nums).all() else np.nan
    rho_loc = spearmanr(filename_rank, slice_locs).correlation if not np.isnan(slice_locs).all() else np.nan
    # InstanceNumber vs SliceLocation deberian estar muy correlacionados
    # entre si (son dos formas de expresar el mismo orden fisico) -
    # si no lo estan, algo raro pasa con esa serie en particular.
    rho_inst_vs_loc = spearmanr(inst_nums, slice_locs).correlation if not (np.isnan(inst_nums).all() or np.isnan(slice_locs).all()) else np.nan
    print(f"{row['StudyInstanceUID'][:16]}.../{row['SeriesInstanceUID'][:16]}...  "
          f"n={len(files):3d}  rho(filename, InstanceNumber)={rho_inst:.3f}  "
          f"rho(filename, SliceLocation)={rho_loc:.3f}  "
          f"rho(InstanceNumber, SliceLocation)={rho_inst_vs_loc:.3f}")


## E. Mezcla de transfer syntaxes

La Dataset Description avisa de sintaxis de transferencia mixta (JPEG Lossless, JPEG 2000, Implicit/Explicit VR Little Endian). Confirmar cuales aparecen y si pydicom decodifica los pixeles de todas sin plugins extra (pylibjpeg/gdcm) antes de asumir que `pydicom.dcmread(...).pixel_array` funciona igual para toda la serie.

In [ ]:
# Solo headers (stop_before_pixels) sobre una muestra de estudios para
# no recorrer el arbol completo de 0.5 TB. Si esto tarda demasiado,
# bajar N_STUDIES.
N_STUDIES = 30
sample_studies = train_series["StudyInstanceUID"].drop_duplicates().sample(n=N_STUDIES, random_state=42)

syntaxes = Counter()
decode_failures = []
for study_id in sample_studies:
    study_dir = RAW_DIR / "train_series" / study_id
    for series_dir in study_dir.iterdir():
        first_file = next(series_dir.glob("*.dcm"), None)
        if first_file is None:
            continue
        ds = pydicom.dcmread(first_file, stop_before_pixels=True)
        tsyn = str(getattr(ds.file_meta, "TransferSyntaxUID", "unknown"))
        syntaxes[tsyn] += 1
        try:
            pydicom.dcmread(first_file).pixel_array  # forces decode
        except Exception as e:
            decode_failures.append((str(first_file), tsyn, str(e)))

print("Transfer syntaxes vistas (UID -> n series):")
for uid, n in syntaxes.most_common():
    print(f"  {uid}: {n}")

print(f"\nFallos al decodificar pixel_array: {len(decode_failures)}")
for path, tsyn, err in decode_failures[:5]:
    print(f"  {path} [{tsyn}]: {err}")


## F. Espaciado de voxel (`PixelSpacing`) — pendiente de la Fase 1

Objetivo (plan Fase 1, ultimo punto): confirmar que `PixelSpacing` (mm/pixel) varia entre estudios/series lo suficiente como para justificar `src/features.py::normalize_physical_scale` (redimensionar por campo de vision fisico, no por pixeles fijos). Tambien mirar `SliceThickness`/`SpacingBetweenSlices` (relevante si el modelo llega a usar contexto entre cortes) y si el espaciado depende del plano (sagital/coronal/axial suelen protocolarse distinto).

In [ ]:
# Un archivo por serie (stop_before_pixels) sobre una muestra de
# estudios, reutilizando el mismo N_STUDIES/sample_studies de la
# seccion E para no recorrer el arbol completo de 0.5 TB otra vez.
rows = []
for study_id in sample_studies:
    study_dir = RAW_DIR / "train_series" / study_id
    for series_dir in study_dir.iterdir():
        first_file = next(series_dir.glob("*.dcm"), None)
        if first_file is None:
            continue
        ds = pydicom.dcmread(first_file, stop_before_pixels=True)
        ps = getattr(ds, "PixelSpacing", None)
        rows.append({
            "StudyInstanceUID": study_id,
            "SeriesInstanceUID": series_dir.name,
            "row_spacing_mm": float(ps[0]) if ps is not None else np.nan,
            "col_spacing_mm": float(ps[1]) if ps is not None else np.nan,
            "Rows": int(getattr(ds, "Rows", np.nan)),
            "Columns": int(getattr(ds, "Columns", np.nan)),
            "SliceThickness": float(getattr(ds, "SliceThickness", np.nan)),
            "SpacingBetweenSlices": float(getattr(ds, "SpacingBetweenSlices", np.nan)),
        })

voxel_df = pd.DataFrame(rows).merge(
    train_series[["StudyInstanceUID", "SeriesInstanceUID", "Anatomical_Plane"]],
    on=["StudyInstanceUID", "SeriesInstanceUID"],
    how="left",
)
# Campo de vision fisico (mm) cubierto por la imagen - lo que
# normalize_physical_scale necesita para redimensionar por mm, no por
# pixeles fijos.
voxel_df["fov_row_mm"] = voxel_df["Rows"] * voxel_df["row_spacing_mm"]
voxel_df["fov_col_mm"] = voxel_df["Columns"] * voxel_df["col_spacing_mm"]

print(f"{len(voxel_df)} series de {voxel_df['StudyInstanceUID'].nunique()} estudios\n")
print("PixelSpacing (mm/pixel):")
print(voxel_df[["row_spacing_mm", "col_spacing_mm"]].describe())
print("\nCampo de vision (mm):")
print(voxel_df[["fov_row_mm", "fov_col_mm"]].describe())
print("\nSliceThickness / SpacingBetweenSlices (mm):")
print(voxel_df[["SliceThickness", "SpacingBetweenSlices"]].describe())

print("\nPixelSpacing por plano (media +- std):")
print(voxel_df.groupby("Anatomical_Plane")[["row_spacing_mm", "col_spacing_mm"]].agg(["mean", "std"]))

# Ratio entre el spacing mas ancho y el mas fino visto - si es grande,
# confirma que un resize a pixeles fijos mezclaria escalas fisicas muy
# distintas (justifica normalize_physical_scale).
ratio = voxel_df["row_spacing_mm"].max() / voxel_df["row_spacing_mm"].min()
print(f"\nRatio row_spacing_mm max/min: {ratio:.2f}x")
